# Lab 15: The Messages List In Depth / GroqClient Wrapper Class

## The capstone

This lab wraps the hand-rolled `messages` list from lab 13 into a `GroqClient` class that owns `self.messages` and grows it automatically inside `send()`. The constructor also takes a `transport` callable (default: a real API call), which is what lets the exercise below test the class with `fake_transport` and zero network access.

## Exercise

**Goal:** write a `GroqClient` class that **owns the chat history**. Chat models forget everything between calls, so every request must contain the whole conversation so far. Instead of managing that list by hand (as in the previous lab), your class keeps it and grows it each time you call `send(...)`. You will test it with a **fake** reply function, so no internet and no API key are needed here.

## Words you will see

- **Class and object**: a class is a blueprint; `GroqClient(...)` builds one object from it. The object keeps its own data in attributes such as `self.messages`.
- **Method**: a function inside a class, called like `client.send("Hi")`. `self` is the object itself.
- **Property**: a method you read like a variable, without brackets. `client.history` is a property (already written for you).
- **State**: data an object remembers between calls. Here, the messages list.
- **Message and role**: a message is a dict like `{"role": "user", "content": "Hi"}`. Roles are `"system"` (rules for the model), `"user"` (the human) and `"assistant"` (the model).
- **Stateless model**: the server remembers nothing, so the full list is sent every call.
- **Transport**: a function with the shape `transport(messages, api_key, model) -> str` that turns a messages list into a reply string. In real use it calls the Groq API. `self.transport` holds it, and you call it like any function.
- **Fake transport**: a stand-in transport that returns canned text with no network. `fake_transport` (already written below) replies `"(fake reply to: <last user text>)"`.
- **Dependency injection**: passing the class the piece it depends on (the transport) instead of hard-wiring it in. That is why tests can use a fake.
- **Copy of a list**: `list(x)` makes a new list, so changing the copy does not change the original.

## What is already written for you

`__init__` stores `api_key`, `model`, `system_prompt` and `transport`, and starts `self.messages` as `[]`, or as `[{"role": "system", "content": system_prompt}]` if a system prompt was given. The `history` property and `fake_transport` are also done. You write four methods.

### `add_user_message(self, text)`

**Receives:** `text` (str). **Returns:** nothing (`None`); it changes `self.messages`.

**Idea:** append one dict with role `"user"` and your text to `self.messages`.

**Check:** on a new client with no system prompt, after `add_user_message("Hi")`, `self.messages == [{"role": "user", "content": "Hi"}]`.

### `add_assistant_message(self, text)`

**Receives:** `text` (str). **Returns:** `None`.

**Idea:** the same as above, but the role is `"assistant"`.

**Check:** after `add_assistant_message("Hello")` on an empty client, `self.messages == [{"role": "assistant", "content": "Hello"}]`.

### `send(self, user_text)`

**Receives:** `user_text` (str), what the human says. **Returns:** `str`, the reply text.

**Idea in plain words:**

1. Add the user's text as a user message (you can reuse `add_user_message`).
2. Call `self.transport(self.messages, self.api_key, self.model)`. The list it receives already ends with the user message. Keep the reply string.
3. Add that reply as an assistant message.
4. Return the reply.

**Worked example** (I ran the reference solution):

```python
client = GroqClient(api_key=None, model="fake-model", system_prompt="You are helpful.", transport=fake_transport)
client.send("Hello!")     # '(fake reply to: Hello!)'
client.history            # [{'role': 'system', 'content': 'You are helpful.'},
                          #  {'role': 'user', 'content': 'Hello!'},
                          #  {'role': 'assistant', 'content': '(fake reply to: Hello!)'}]
```

**Check:** after one `send`, `len(client.history)` is 3 (system, user, assistant). After two sends it is 5. Without a system prompt it would be 2 and 4.

### `reset(self)`

**Receives:** nothing. **Returns:** `None`.

**Idea:** throw away the conversation but keep the rules. If `self.system_prompt` was set, `self.messages` becomes a new list holding just the system message. If there is no system prompt, it becomes an empty list `[]`.

**Check:** after two sends and a `reset()` on the example client, `client.history` is `[{"role": "system", "content": "You are helpful."}]`. On a client with no system prompt, `client.history` is `[]`.

### What the checks below test

The cell creates a client with `fake_transport`, then asserts that: the starting history is just the system message; `send("Hello!")` returns `"(fake reply to: Hello!)"`; the history afterwards is system, user, assistant in that order; after a second send the history has 5 messages; and after `reset()` it is back to just the system message. When all asserts pass, the cell prints **All checks passed**. That shows your message-list logic is correct without any network access.

In [ ]:
class GroqClient:
    def __init__(self, api_key: str | None, model: str, system_prompt: str | None = None, transport=None):
        self.api_key = api_key
        self.model = model
        self.system_prompt = system_prompt
        self.transport = transport
        self.messages: list[dict] = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def add_user_message(self, text: str) -> None:
        """Add one user message to the conversation.

        Args:
            text: what the human said, for example "Hi".

        Returns:
            None. It changes self.messages.

        Example:
            on an empty client, add_user_message("Hi") leaves
            self.messages == [{"role": "user", "content": "Hi"}]

        TODO:
            1. Build a dict with "role" set to "user" and "content" set to text.
            2. Append it to self.messages.
        """
        raise NotImplementedError("GroqClient.add_user_message() is not implemented yet - replace the TODO with your code")

    def add_assistant_message(self, text: str) -> None:
        """Add one assistant (model) message to the conversation.

        Args:
            text: what the model said, for example "Hello".

        Returns:
            None. It changes self.messages.

        Example:
            on an empty client, add_assistant_message("Hello") leaves
            self.messages == [{"role": "assistant", "content": "Hello"}]

        TODO:
            1. Build a dict with "role" set to "assistant" and "content" set to text.
            2. Append it to self.messages.
        """
        raise NotImplementedError("GroqClient.add_assistant_message() is not implemented yet - replace the TODO with your code")

    def send(self, user_text: str) -> str:
        """Send one user message and get the model's reply, keeping the history up to date.

        Args:
            user_text: what the human says, for example "Hello!".

        Returns:
            str: the reply text from the transport.

        Example:
            with fake_transport, client.send("Hello!") returns
            "(fake reply to: Hello!)" and the history grows by 2 messages
            (the user message, then the assistant reply).

        TODO:
            1. Add user_text as a user message.
            2. Call self.transport(self.messages, self.api_key, self.model)
               and keep the reply string.
            3. Add the reply as an assistant message.
            4. Return the reply.
        """
        raise NotImplementedError("GroqClient.send() is not implemented yet - replace the TODO with your code")

    def reset(self) -> None:
        """Forget the conversation but keep the system prompt.

        Args:
            None (uses self.system_prompt).

        Returns:
            None. It replaces self.messages.

        Example:
            with system_prompt "You are helpful.", after any number of sends,
            reset() leaves self.messages ==
            [{"role": "system", "content": "You are helpful."}].
            With no system prompt it leaves [].

        TODO:
            1. If self.system_prompt is set, make self.messages a new list
               holding only the system message.
            2. Otherwise make self.messages an empty list.
        """
        raise NotImplementedError("GroqClient.reset() is not implemented yet - replace the TODO with your code")

    @property
    def history(self) -> list[dict]:
        return list(self.messages)


def fake_transport(messages: list[dict], api_key: str | None, model: str) -> str:
    """A stand-in for a real API call - lets us test GroqClient with zero network access."""
    last_user_message = messages[-1]["content"]
    return f"(fake reply to: {last_user_message})"


client = GroqClient(api_key=None, model="fake-model", system_prompt="You are helpful.", transport=fake_transport)
assert client.history == [{"role": "system", "content": "You are helpful."}], 'client.history should be ' + repr([{"role": "system", "content": "You are helpful."}]) + ", got " + repr(client.history)

reply = client.send("Hello!")
assert reply == "(fake reply to: Hello!)", 'reply should be ' + repr("(fake reply to: Hello!)") + ", got " + repr(reply)
assert client.history == [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": reply},
], 'client.history should be ' + repr([
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": reply},
]) + ", got " + repr(client.history)

client.send("Second message")
assert len(client.history) == 5, 'len(client.history) should be ' + repr(5) + ", got " + repr(len(client.history))

client.reset()
assert client.history == [{"role": "system", "content": "You are helpful."}], 'client.history should be ' + repr([{"role": "system", "content": "You are helpful."}]) + ", got " + repr(client.history)

print("All checks passed - GroqClient's message-list logic works with zero network access.")

## Solution

In [ ]:
class GroqClient:
    def __init__(self, api_key: str | None, model: str, system_prompt: str | None = None, transport=None):
        self.api_key = api_key
        self.model = model
        self.system_prompt = system_prompt
        self.transport = transport
        self.messages: list[dict] = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def add_user_message(self, text: str) -> None:
        self.messages.append({"role": "user", "content": text})

    def add_assistant_message(self, text: str) -> None:
        self.messages.append({"role": "assistant", "content": text})

    def send(self, user_text: str) -> str:
        self.add_user_message(user_text)
        reply = self.transport(self.messages, self.api_key, self.model)
        self.add_assistant_message(reply)
        return reply

    def reset(self) -> None:
        self.messages = [{"role": "system", "content": self.system_prompt}] if self.system_prompt else []

    @property
    def history(self) -> list[dict]:
        return list(self.messages)


client = GroqClient(api_key=None, model="fake-model", system_prompt="You are helpful.", transport=fake_transport)
print(client.send("Hello!"))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Rebuilds lab 13's digital twin on top of `GroqClient` instead of a hand-rolled messages list, using a real transport this time. Fill in `my_profile` with your own bio, same as lab 13.

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


In [ ]:
def build_digital_twin_system_prompt(profile: dict) -> str:
    lines = [
        f"You are role-playing as {profile['name']}, {profile['role']}.",
        f"Speak in a tone that is {profile['tone']}.",
        "Facts you can draw on about this person:",
    ]
    lines += [f"- {fact}" for fact in profile["facts"]]
    lines.append("Boundaries you must respect:")
    lines += [f"- {b}" for b in profile["boundaries"]]
    return "\n".join(lines)


def real_transport(messages: list[dict], api_key: str | None, model: str) -> str:
    return call_groq(messages, api_key=api_key, model=model)


my_profile = {
    "name": "your name",
    "role": "however you'd describe your role in one clause",
    "tone": "a few adjectives for how you talk",
    "facts": ["a real fact about you", "another real fact about you"],
    "boundaries": ["something this digital twin should always refuse to do"],
}

twin = GroqClient(
    api_key=GROQ_API_KEY,
    model=DEFAULT_MODEL,
    system_prompt=build_digital_twin_system_prompt(my_profile),
    transport=real_transport,
)

for user_message in ["Hi! What are you working on lately?", "What's the most useful thing you've learned recently?"]:
    reply = twin.send(user_message)
    print(f"You: {user_message}\nTwin: {reply}\n")

print("--- Full messages list ---")
for message in twin.history:
    print(f"{message['role']:<10} {message['content'][:80]}")